# qv56 input4MIPs discovery

Scratch notebook: what forcing data is actually available on Gadi, and is it
enough for (a) the CEDS SLCF/CO2 historical evaluation and (b) a possible
ScenarioMIP extension.

Nothing here is written to the repo. Read-only exploration.

In [1]:
from pathlib import Path
import os

CANDIDATES = [
    Path("/g/data/qv56/replicas/input4MIPs"),
    Path("/g/data/qv56/replicas"),
    Path("/g/data/qv56"),
]

ROOT = None
for path in CANDIDATES:
    try:
        if path.is_dir():
            ROOT = path
            break
    except PermissionError:
        print(f"PermissionError: {path} — is qv56 in your storage flags?")

print(f"Root: {ROOT}\n")
if ROOT is not None:
    for entry in sorted(ROOT.iterdir())[:40]:
        print(f"  {'d' if entry.is_dir() else 'f'}  {entry.name}")

Root: /g/data/qv56/replicas/input4MIPs

  d  CMIP6
  d  CMIP6Plus
  d  CMIP7
  d  CMIP7Plus


## 1. Walk the DRS

input4MIPs follows a Data Reference Syntax path convention, roughly

    <mip_era>/<activity_id>/<institution_id>/<source_id>/<realm>/<frequency>/
    <variable_id>/<grid_label>/<version>/

but replicas vary. Walk a bounded depth and infer the structure rather than
assuming it.

In [2]:
import itertools

def walk_levels(root: Path, max_depth: int = 4, max_per_level: int = 25):
    """Print directory names level by level, without descending into file-heavy dirs."""
    level = [root]
    for depth in range(max_depth):
        names, nxt = set(), []
        for directory in level[:60]:
            try:
                for entry in itertools.islice(directory.iterdir(), 400):
                    if entry.is_dir():
                        names.add(entry.name)
                        nxt.append(entry)
            except (PermissionError, OSError):
                continue
        if not names:
            break
        shown = sorted(names)[:max_per_level]
        more = f"  (+{len(names) - len(shown)} more)" if len(names) > len(shown) else ""
        print(f"depth {depth + 1}: {len(names)} unique{more}")
        print("   " + ", ".join(shown) + "\n")
        level = nxt

walk_levels(ROOT)

depth 1: 4 unique
   CMIP6, CMIP6Plus, CMIP7, CMIP7Plus

depth 2: 13 unique
   AerChemMIP, AerChemMIP2, C4MIP, CMIP, DAMIP, DCPP, FAFMIP, HighResMIP, ISMIP6, OMIP, PAMIP, RFMIP, ScenarioMIP

depth 3: 28 unique  (+3 more)
   CCCma, CNRM-Cerfacs, CR, DRES, FZJ, IACETH, IAMC, IIASA-IAMC, ImperialCollege, MOHC, MPI-B, MPI-M, MRI, NASA-GSFC, NCAR, NCAS, PCMDI, PIK, PNNL-JGCRI, SOLARIS-HEPPA, UCI, UCLA, UColorado, UReading, UoM

depth 4: 340 unique  (+315 more)
   ACCESS1-3-rcp85-1-0, CCMI-hist-nat-1-1, CCMI-hist-sol-1-1, CCMI-hist-stratO3-1-0, CCMI-hist-volc-1-1, CCMI-ssp245-nat-1-0, CCMI-ssp245-sol-1-0, CCMI-ssp245-stratO3-1-0, CCMI-ssp245-volc-1-0, CCSM4-rcp26-1-0, CCSM4-rcp85-1-0, CEDS-2016-06-18, CEDS-2016-06-18-sectorDimV2, CEDS-2016-06-18-supplemental-data, CEDS-2016-07-26, CEDS-2016-07-26-sectorDim, CEDS-2016-07-26-sectorDim-supplemental-data, CEDS-2017-05-18, CEDS-2017-05-18-supplemental-data, CEDS-2017-08-30, CEDS-2017-08-30-supplemental-data, CEDS-2017-10-05, CEDS-CMIP-2024-10-21,

## 2. Is CMIP7 CEDS present?

The historical evaluation needs `CEDS-CMIP-2025-04-18` (or a later CMIP7 CEDS
source id). Search for any CEDS directory regardless of depth or era.

In [3]:
import re
import subprocess

# `find` is far faster than pathlib on Lustre for a sparse search.
result = subprocess.run(
    ["find", str(ROOT), "-maxdepth", "6", "-type", "d", "-iname", "*CEDS*"],
    capture_output=True, text=True, timeout=600,
)
ceds_dirs = sorted(set(result.stdout.split()))

print(f"{len(ceds_dirs)} CEDS directories\n")
for d in ceds_dirs[:40]:
    print("  " + d.replace(str(ROOT) + "/", ""))

source_ids = sorted({
    m.group(1) for d in ceds_dirs
    if (m := re.search(r"(CEDS[-\w]*?\d{4}-\d{2}-\d{2}|CEDS-\d{4}-\d{2}-\d{2}|CEDS[\w-]*)", d))
})
print(f"\nDistinct CEDS source ids: {source_ids}")

19 CEDS directories

  CMIP6/CMIP/PNNL-JGCRI/CEDS-2016-06-18
  CMIP6/CMIP/PNNL-JGCRI/CEDS-2016-06-18-sectorDimV2
  CMIP6/CMIP/PNNL-JGCRI/CEDS-2016-06-18-supplemental-data
  CMIP6/CMIP/PNNL-JGCRI/CEDS-2016-07-26
  CMIP6/CMIP/PNNL-JGCRI/CEDS-2016-07-26-sectorDim
  CMIP6/CMIP/PNNL-JGCRI/CEDS-2016-07-26-sectorDim-supplemental-data
  CMIP6/CMIP/PNNL-JGCRI/CEDS-2017-05-18
  CMIP6/CMIP/PNNL-JGCRI/CEDS-2017-05-18-supplemental-data
  CMIP6/CMIP/PNNL-JGCRI/CEDS-2017-08-30
  CMIP6/CMIP/PNNL-JGCRI/CEDS-2017-08-30-supplemental-data
  CMIP6/CMIP/PNNL-JGCRI/CEDS-2017-10-05
  CMIP6Plus/CMIP/PNNL-JGCRI/CEDS-CMIP-2024-10-21
  CMIP6Plus/CMIP/PNNL-JGCRI/CEDS-CMIP-2024-10-21-supplemental
  CMIP6Plus/CMIP/PNNL-JGCRI/CEDS-CMIP-2024-11-25
  CMIP6Plus/CMIP/PNNL-JGCRI/CEDS-CMIP-2024-11-25-supplemental
  CMIP7/CMIP/PNNL-JGCRI/CEDS-CMIP-2025-03-18
  CMIP7/CMIP/PNNL-JGCRI/CEDS-CMIP-2025-03-18-supplemental
  CMIP7/CMIP/PNNL-JGCRI/CEDS-CMIP-2025-04-18
  CMIP7/CMIP/PNNL-JGCRI/CEDS-CMIP-2025-04-18-supplemental

Distin

## 3. Inventory whatever CEDS exists

Species, grid labels, time coverage, versions and total size — enough to decide
whether the local 47 GB download is redundant.

In [7]:
import subprocess
import pandas as pd
from pathlib import Path

CEDS_ROOT = ROOT / "CMIP7" / "CMIP" / "PNNL-JGCRI"

result = subprocess.run(
    ["find", str(CEDS_ROOT), "-name", "*.nc", "-type", "f"],
    capture_output=True, text=True, timeout=900,
)
paths = [Path(p) for p in result.stdout.split()]
print(f"{len(paths)} NetCDF files under {CEDS_ROOT}\n")

rows = []
for p in paths:
    rel = p.relative_to(ROOT).parts
    rows.append({
        "source_id": rel[3],
        "realm": rel[4] if len(rel) > 4 else None,
        "frequency": rel[5] if len(rel) > 5 else None,
        "variable_id": rel[6] if len(rel) > 6 else None,
        "grid_label": rel[7] if len(rel) > 7 else None,
        "version": rel[8] if len(rel) > 8 else None,
        "file": p.name,
        "dirpath": str(p.parent),
        "size_gb": p.stat().st_size / 1e9,
    })

inv = pd.DataFrame(rows)
inv["species"] = inv["file"].str.split("-em-").str[0]
inv["stream"] = inv["file"].str.contains("-em-AIR-").map({True: "AIR", False: "surface"})

print(
    inv.groupby(["source_id", "grid_label", "version"])
    .agg(n_files=("file", "count"), gb=("size_gb", "sum"))
    .round(1).to_string()
)

2062 NetCDF files under /g/data/qv56/replicas/input4MIPs/CMIP7/CMIP/PNNL-JGCRI

                                                        n_files     gb
source_id                         grid_label version                  
CEDS-CMIP-2025-03-18              gn         v20250325       97   47.4
                                             v20250327        8    4.1
CEDS-CMIP-2025-03-18-supplemental gn         v20250325      200   81.3
CEDS-CMIP-2025-04-18              gn         v20250421      105   51.5
                                  gr         v20250610      440   84.3
CEDS-CMIP-2025-04-18-supplemental gn         v20250421      200   81.3
                                  gr         v20250610     1012  163.2


In [8]:
april = inv[(inv["source_id"] == "CEDS-CMIP-2025-04-18") & (inv["grid_label"] == "gn")]

print(f"{len(april)} gn files, {april['size_gb'].sum():.1f} GB")
print(f"Versions: {sorted(april['version'].unique())}\n")
print(
    april.groupby(["stream", "species"])
    .agg(n=("file", "count"), dirs=("dirpath", "nunique"))
    .to_string()
)
print(f"\nDistinct leaf directories: {april['dirpath'].nunique()}")

105 gn files, 51.5 GB
Versions: ['v20250421']

                                                                        n  dirs
stream  species                                                                
AIR     BC                                                              6     1
        CH4                                                             2     1
        CO                                                              6     1
        CO2                                                             6     1
        N2O                                                             2     1
        NH3                                                             6     1
        NMVOC                                                           6     1
        NOx                                                             6     1
        OC                                                              6     1
        SO2                                                             6

In [9]:
for pattern in ["*UOEXETER*", "*EVA*", "*aerosol-optical*", "*stratO3*"]:
    out = subprocess.run(["find", str(ROOT), "-maxdepth", "4", "-type", "d", "-iname", pattern],
                         capture_output=True, text=True, timeout=300)
    print(f"{pattern}: {[h.replace(str(ROOT)+'/','') for h in out.stdout.split()]}")

*UOEXETER*: ['CMIP6Plus/CMIP/uoexeter', 'CMIP6Plus/CMIP/uoexeter/UOEXETER-CMIP-1-3-1', 'CMIP6Plus/CMIP/uoexeter/UOEXETER-CMIP-1-2-0', 'CMIP6Plus/CMIP/uoexeter/UOEXETER-CMIP-1-1-3', 'CMIP6Plus/CMIP/uoexeter/UOEXETER-CMIP-1-3-0', 'CMIP7/CMIP/uoexeter', 'CMIP7/CMIP/uoexeter/UOEXETER-CMIP-2-0-0', 'CMIP7/CMIP/uoexeter/UOEXETER-CMIP-2-2-1', 'CMIP7/ScenarioMIP/uoexeter', 'CMIP7/ScenarioMIP/uoexeter/UOEXETER-ScenarioMIP-2-2-2', 'CMIP7/ScenarioMIP/uoexeter/UOEXETER-ScenarioMIP-2-2-1']
*EVA*: []
*aerosol-optical*: []
*stratO3*: ['CMIP6/DAMIP/CCCma/CCMI-ssp245-stratO3-1-0', 'CMIP6/DAMIP/CCCma/CCMI-hist-stratO3-1-0']


## 4. ScenarioMIP and companion forcings

For the optional scenario extension, and for the solar/volcanic inputs the
all-forcing FaIR run needs regardless.

In [5]:
TARGETS = {
    "ScenarioMIP": ["*ScenarioMIP*", "*scenario*"],
    "solar":       ["*solar*", "*SOLARIS*"],
    "volcanic":    ["*volcan*", "*CMIP-volc*"],
    "GHG conc":    ["*GHGConcentration*", "*UoM*"],
    "biomass":     ["*BB4CMIP*", "*GFED*"],
    "aerosol opt": ["*aerosol*", "*MACv2*"],
}

for label, patterns in TARGETS.items():
    hits = set()
    for pattern in patterns:
        out = subprocess.run(
            ["find", str(ROOT), "-maxdepth", "5", "-type", "d", "-iname", pattern],
            capture_output=True, text=True, timeout=300,
        )
        hits.update(out.stdout.split())
    print(f"\n{label}: {len(hits)} dir(s)")
    for h in sorted(hits)[:6]:
        print("   " + h.replace(str(ROOT) + "/", ""))


ScenarioMIP: 7 dir(s)
   CMIP6/ScenarioMIP
   CMIP6Plus/ScenarioMIP
   CMIP6Plus/ScenarioMIP/SOLARIS-HEPPA/SOLARIS-HEPPA-ScenarioMIP-4-6-a002
   CMIP7/ScenarioMIP
   CMIP7/ScenarioMIP/SOLARIS-HEPPA/SOLARIS-HEPPA-ScenarioMIP-4-6
   CMIP7/ScenarioMIP/uoexeter/UOEXETER-ScenarioMIP-2-2-1

solar: 10 dir(s)
   CMIP6/CMIP/SOLARIS-HEPPA
   CMIP6/CMIP/SOLARIS-HEPPA/SOLARIS-HEPPA-3-2
   CMIP6Plus/CMIP/SOLARIS-HEPPA
   CMIP6Plus/CMIP/SOLARIS-HEPPA/SOLARIS-HEPPA-CMIP-4-5
   CMIP6Plus/ScenarioMIP/SOLARIS-HEPPA
   CMIP6Plus/ScenarioMIP/SOLARIS-HEPPA/SOLARIS-HEPPA-ScenarioMIP-4-6-a002

volcanic: 0 dir(s)

GHG conc: 23 dir(s)
   CMIP6/AerChemMIP/UoM
   CMIP6/AerChemMIP/UoM/UoM-AIM-ssp370-lowNTCF-1-2-0
   CMIP6/AerChemMIP/UoM/UoM-AIM-ssp370-lowNTCF-1-2-1
   CMIP6/CMIP/UoM
   CMIP6/CMIP/UoM/UoM-CMIP-1-2-0
   CMIP6/ScenarioMIP/UoM

biomass: 6 dir(s)
   CMIP6/CMIP/VUA/VUA-CMIP-BB4CMIP6-1-0
   CMIP6/CMIP/VUA/VUA-CMIP-BB4CMIP6-1-1
   CMIP6/CMIP/VUA/VUA-CMIP-BB4CMIP6-1-2
   CMIP6Plus/CMIP/DRES/DRES-CMIP-BB4